# Custom tools

In [1]:
# step 1 : create a function
def multiply(a, b):
    """Multiply two numbers"""
    return a * b

In [2]:
# step 2 : add type hints

def multiply(a : int, b : int) -> int:
    """Multiply two numbers"""
    return a * b

In [4]:
!pip install langchain-core

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: C:\Program Files\Python313\python.exe -m pip install --upgrade pip


In [3]:
# step 3 : add tool decorator
from langchain_core.tools import tool

@tool
def multiply(a : int, b : int) -> int:
    """Multiply two numbers"""
    return a * b

ModuleNotFoundError: No module named 'langchain_core'

In [4]:
result = multiply.invoke({"a" : 3, "b" : 5})
print(result)

15


In [5]:
print(multiply.name) # tool name

multiply


In [6]:
print(multiply.description) # description about tool

Multiply two numbers


In [7]:
print(multiply.args) # args name

{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [8]:
# llm sees the json schema for this tool
print(multiply.args_schema.model_json_schema())

{'description': 'Multiply two numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


In [9]:
import json
with open("tool_schema.json", "w") as f:
    json.dump(multiply.args_schema.model_json_schema(), f)

In [10]:
with open("tool_schema.json", "r") as f:
    json_schema = json.load(f)

In [11]:
print(json_schema)

{'description': 'Multiply two numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


In [12]:
# using structured and pydantic tools
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

In [13]:
class Multiply_Input(BaseModel):
    a : int = Field(required = True, description = "First no for multiplication : ")
    b : int = Field(required = True, description = "Second no for multiplication : ")

C:\Users\Admin\AppData\Local\Temp\ipykernel_1404\774385468.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a : int = Field(required = True, description = "First no for multiplication : ")
C:\Users\Admin\AppData\Local\Temp\ipykernel_1404\774385468.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b : int = Field(required = True, description = "Second no for multiplication : ")


In [14]:
def multiply(a, b):
    return a * b

In [15]:
multiply_tool = StructuredTool.from_function(
    func = multiply, # function,
    name = "multiply",
    description = "Multiplies two numbers",
    args_schema = Multiply_Input
)

In [16]:
multiply_tool.args_schema.model_json_schema()

{'properties': {'a': {'description': 'First no for multiplication : ',
   'required': True,
   'title': 'A',
   'type': 'integer'},
  'b': {'description': 'Second no for multiplication : ',
   'required': True,
   'title': 'B',
   'type': 'integer'}},
 'required': ['a', 'b'],
 'title': 'Multiply_Input',
 'type': 'object'}

In [17]:
multiply_tool.invoke({"a" : 2, "b" : 4})

8

In [18]:
# Using Base Tool
from langchain_core.tools import BaseTool
from typing import Type

In [19]:
class Multiply_Input(BaseModel):
    a : int = Field(required = True, description = "First no for multiplication : ")
    b : int = Field(required = True, description = "Second no for multiplication : ")

C:\Users\Admin\AppData\Local\Temp\ipykernel_1404\774385468.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a : int = Field(required = True, description = "First no for multiplication : ")
C:\Users\Admin\AppData\Local\Temp\ipykernel_1404\774385468.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b : int = Field(required = True, description = "Second no for multiplication : ")


In [20]:
class Multiply_tool(BaseTool):
    name : str = 'multiply'
    description : str = "Multiply two numbers together"
    args_schema : Type[BaseModel] = Multiply_Input
    

    def _run(self, a : int, b : int) -> int:
        return a * b

In [21]:
tool = Multiply_tool()

In [22]:
tool.invoke({"a" : 3, "b" : 10})

30

In [23]:
print("hello")

hello


In [24]:
from langchain_core.tools import tool

# custom tool
@tool
def add(a:int, b:int) -> int:
    "return the sum of two numbers"
    return a + b

@tool
def multiply(a:int, b:int) ->int:
    "return the multiplication of two numbers"
    return a*b

In [25]:
class MathToolkit:
    def get_tools(self):
        return [add, multiply]

In [26]:
toolkit = MathToolkit()

In [27]:
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name , "=>", tool.description)

add => return the sum of two numbers
multiply => return the multiplication of two numbers


In [28]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
import warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv
load_dotenv()

# Initialize the chat model
llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",  # Hugging Face model repo
    task = "text-generation",
    temperature = 0,
    max_new_tokens= 1000,
)

model = ChatHuggingFace(llm = llm)

In [29]:
llm_tools = model.bind_tools([multiply])

In [30]:
llm_tools.invoke("Hi how are you?").content

HfHubHTTPError: Client error '401 Unauthorized' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a7480f3-43c6cd2c099b733d7eb9a19b;4ef35c65-f8c1-4227-8953-2ff2af397921)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.

In [ ]:
llm_tools.invoke("can you multiply 3 with 10 ")

AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":3,"b":10}', 'name': 'multiply', 'description': None}, 'id': 'd7bf5cd5c', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 132, 'total_tokens': 198}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8a76e344b64d754137e2', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f314a-8d52-78b3-b081-4aec6eb431f9-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'd7bf5cd5c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 132, 'output_tokens': 66, 'total_tokens': 198})

In [ ]:
from langchain_core.messages import HumanMessage

In [ ]:
query = HumanMessage("can you multiply 3 with 100")
messages = [query]

In [ ]:
result = llm_tools.invoke(messages)

In [ ]:
result

AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":3,"b":100}', 'name': 'multiply', 'description': None}, 'id': '976dbcb52', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 131, 'total_tokens': 172}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8a76e344b64d754137e2', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f314b-d9bd-7193-a864-beb04d12cecf-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 100}, 'id': '976dbcb52', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 131, 'output_tokens': 41, 'total_tokens': 172})

In [ ]:
messages.append(result)

In [ ]:
messages

[HumanMessage(content='can you multiply 3 with 100', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":3,"b":100}', 'name': 'multiply', 'description': None}, 'id': '976dbcb52', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 131, 'total_tokens': 172}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8a76e344b64d754137e2', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f314b-d9bd-7193-a864-beb04d12cecf-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 100}, 'id': '976dbcb52', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 131, 'output_tokens': 41, 'total_tokens': 172})]

In [ ]:
tool_result = multiply.invoke(result.tool_calls[0])

In [ ]:
tool_result

ToolMessage(content='300', name='multiply', tool_call_id='976dbcb52')

In [ ]:
messages.append(tool_result)

In [ ]:
messages

[HumanMessage(content='can you multiply 3 with 100', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":3,"b":100}', 'name': 'multiply', 'description': None}, 'id': '976dbcb52', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 131, 'total_tokens': 172}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8a76e344b64d754137e2', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f314b-d9bd-7193-a864-beb04d12cecf-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 100}, 'id': '976dbcb52', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 131, 'output_tokens': 41, 'total_tokens': 172}),
 ToolMessage(content='300', name='multiply', tool_call_id='976dbcb52')]

In [ ]:
llm_tools.invoke(messages).content

'The product of 3 and 100 is **300**.'